# ISOM 835 · The Tabular Frontier — Opening Day
**Suffolk University · Sawyer Business School · Fall 2026 · Prof. Hasan Arslan**

**Tonight in one sentence:** every model in this course starts from zero, so it needs **a lot** of data. In 2025–26 a new kind of model arrived that has **already "read" millions of tables**, so it can do well with **very little** data, and it needs **no tuning at all**.

**Before you start (one time):** in Colab, click **Runtime → Change runtime type → T4 GPU → Save**. The new model runs much faster on a GPU. Everything else works without one.

**How to use this notebook:** run the cells top to bottom (Shift + Enter). Read the short text above each cell first: it tells you **what the cell does** and **how to read the output**.

### Tonight's map

| Step | The question | What you will see |
|---|---|---|
| **1** | How much data does our model need? | the **Opening Day** problem: a new hotel with only a few hundred bookings |
| **2** | Where does small data show up in business? | almost everywhere |
| **3** | What is a tabular foundation model? | the fresh graduate and the experienced manager |
| **4** | How can a model "read" tables before it sees ours? | its textbooks: **made-up tables**. Try one yourself |
| **5** | How does it predict without training? | your rows become the **prompt** |
| **6** | Does it really help on Opening Day? | **the race**, on our hotel bookings |
| **7** | So which one do I use? | a decision table and five business cases |
| **8** | Your turn | three questions |

In [ ]:
# Setup, part 1 — install TabICL, an open-source tabular foundation model (about a minute)
!pip install -q tabicl

## The business situation: Opening Day

Our hotel group is opening a **new hotel**. After its first few weeks it has only **a few hundred bookings** in its records.

The revenue manager asks the same question as always: *"Which of these new bookings will cancel?"* The difference is that the model now has very little to learn from.

The next cell loads the same 30,000 bookings and the same train/test split as Sessions 8 and 9, so every score tonight can be compared with last week's **0.945**.

In [ ]:
# Setup, part 2 — the same bookings, split and prep station as Sessions 8 and 9
import time, warnings
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/hotel_bookings.csv'
raw = pd.read_csv(URL)
LEAKS = ['reservation_status', 'reservation_status_date', 'assigned_room_type']   # Session 7: these columns are cheating
bookings = raw.sample(30_000, random_state=835)
y = bookings['is_canceled']
X_raw = bookings.drop(columns=['is_canceled'] + LEAKS)
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X_raw, y, test_size=0.2, stratify=y, random_state=835)

numbers = X_raw.select_dtypes(include='number').columns.tolist()
text = [c for c in X_raw.columns if c not in numbers]
prep = ColumnTransformer([
    ('numbers', SimpleImputer(strategy='constant', fill_value=-1), numbers),
    ('text', Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing')),
                       ('codes', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), text),
], verbose_feature_names_out=False).set_output(transform='pandas')
X_tr = prep.fit_transform(X_tr_raw)
X_te = prep.transform(X_te_raw)

def new_boost():
    # Session 8's recipe: gradient boosting with early stopping
    return HistGradientBoostingClassifier(max_iter=3000, early_stopping=True, validation_fraction=0.15, n_iter_no_change=50, random_state=835)

def first_bookings(n, sample=0):
    # A random set of n training bookings, like the first n bookings of a new hotel
    if n >= len(X_tr): return X_tr, y_tr
    rows = train_test_split(X_tr.index, train_size=n, stratify=y_tr, random_state=835 + sample)[0]
    return X_tr.loc[rows], y_tr.loc[rows]

print(f'{len(X_tr):,} bookings to learn from, {len(X_te):,} held back as the exam · {y_tr.mean():.0%} canceled')

## 1. How much data does our model need?

Last week's champion learned from **24,000** bookings and scored **0.945** (AUC: *pick one booking that canceled and one that didn't; how often does the model give the canceler the higher score?* **0.5** = coin flip, **1.0** = perfect).

The new hotel does not have 24,000 bookings. So let's pretend. The next cell gives the **same model** only **50** bookings, then **100**, **200**, **500**… and grades every version on the **same 6,000-booking exam**. For the small sizes it tries 3 different random sets of bookings and averages them, because (remember Session 9) one small sample is partly luck.

It trains about 25 models, which takes only a few seconds.

In [ ]:
sizes = [50, 100, 200, 500, 1000, 2000, 5000, 24000]
boost_curve = {}
start = time.time()
for n in sizes:
    tries = 3 if n <= 2000 else 1
    scores = []
    for sample in range(tries):
        X_small, y_small = first_bookings(n, sample)
        model = new_boost().fit(X_small, y_small)
        scores.append(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]))
    boost_curve[n] = np.mean(scores)
    print(f'{n:>6,} bookings to learn from  →  test AUC {boost_curve[n]:.3f}')
print(f'({time.time() - start:.0f} seconds)')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, [boost_curve[n] for n in sizes], 'o-', color='#5b6bd8', lw=2.5, ms=8, label='gradient boosting (starts from zero)')
ax.axhline(0.5, color='grey', ls=':', lw=1.5); ax.text(55, 0.51, 'a coin flip', color='grey')
ax.axvspan(40, 600, color='#f0a030', alpha=0.12); ax.text(60, 0.94, 'Opening Day:\na new hotel lives here', color='#b06f00', va='top')
ax.set_xscale('log'); ax.set_xticks(sizes); ax.set_xticklabels([f'{n:,}' for n in sizes])
ax.set_ylim(0.45, 1.0); ax.set_xlabel('bookings the model learned from (log scale)'); ax.set_ylabel('test AUC')
ax.set_title('Small data hurts: the same model, fed fewer and fewer bookings'); ax.legend(loc='lower right'); plt.show()

**How to read it:** each dot is the same model recipe, fed a different number of bookings. Left = little data. Right = a lot. Higher = better.

**Stop and notice:**
- With **24,000** bookings: about **0.945**, last week's champion.
- With **5,000**: still good, about **0.93**.
- With **200**, like our new hotel: about **0.80**. With **50**: below **0.70**, much closer to a coin flip.
- The model needs **thousands** of bookings before it is really good. On Opening Day, it does not have them.

**Why?** Gradient boosting **starts from zero**. Everything it knows about cancellations, it must discover from *your* rows. With 200 rows there just isn't much to discover from.

## 2. Small data is everywhere in business

Our hotel story is not unusual. Small tables are the normal case in business, not the exception:

| Situation | Rows you have |
|---|---|
| a **new store, new hotel, new branch** | its first weeks of customers |
| a **new product** launch | the first few hundred buyers |
| a **B2B company** predicting which clients will leave | maybe 200 clients |
| **rare events**: fraud, machine failures, defaults | a few hundred examples of the event |
| a **pilot** or a **survey** | a few hundred responses |
| an **expensive label** (a lawyer, a doctor, an inspector must say yes/no) | as many as you can afford |

In every one of these, the question is the same as on Opening Day: *how do I get a good model from little data?*

## 3. The idea: hire the experienced manager

Imagine you need a manager for the new hotel. Two candidates:

- 🎓 **The fresh graduate** is smart but has never worked in a hotel. Everything they know about cancellations, they will learn from *this* hotel's first 200 bookings.
- 🧳 **The experienced manager** has worked in hundreds of hotels before. With the same 200 bookings, they recognize patterns right away: *"Booked a year ahead, no special requests, through an online agency? I've seen that before."*

On Opening Day, who do you want?

**Gradient boosting is the fresh graduate.** It starts from zero every time.

You already know a model that does **not** start from zero: **ChatGPT**. It read a large part of the internet first. That is a **foundation model**: trained once on a huge amount of data, then reused for many new tasks.

**Since 2025 the same idea works for tables.** *Tabular* just means rows and columns, like a spreadsheet. A **tabular foundation model** has read **millions of tables** before it ever sees yours. It is the experienced manager.

The two best-known ones:

| Model | From | What to know |
|---|---|---|
| **TabICL** | Inria (France), open source | free to use, also commercially. **The one we run tonight.** |
| **TabPFN** | Prior Labs (Germany) | the pioneer (first version 2022); the newest, TabPFN-3.5 (Sept 2026), is recommended by its makers for up to a million rows. Its newest weights are **free for research only**: business use needs a paid license. |

On public benchmarks these models now often **match or beat a carefully tuned boosting model, with zero tuning**.

## 4. Its textbooks: made-up tables

Wait. Where would a model find **millions of tables** to read? Real business tables are private, and there are not millions of them.

The trick: **the tables are made up.** A computer program invents tables endlessly. Each one has a few columns, an answer column, and a **hidden rule** that connects them, plus some noise, like real life. Because the program invented the rule, it always knows the right answers.

Then the model practices one exercise, millions of times: ***"Here are some rows with their answers. What is the answer for this new row?"*** Table after table, rule after rule, it gets good at the **skill itself**: finding the pattern in **any** table quickly.

That is the key idea. It did not learn hotel facts. **It learned how to learn from a table.**

The next cell invents one such table, just like the model's textbooks. Your job: **be the model**. Look at the rows with answers, then guess the answer for the last row.

In [ ]:
def made_up_table(seed, rows=9):
    # Invents a small table: 3 columns, a hidden rule, a little noise. Like one page of the model's textbook.
    rng = np.random.default_rng(seed)
    X = rng.integers(0, 10, size=(rows, 3))
    rule = ['answer is 1 when A + B is big', 'answer is 1 when C is small', 'answer is 1 when A is bigger than B'][seed % 3]
    if seed % 3 == 0: y = (X[:, 0] + X[:, 1] >= 10).astype(int)
    elif seed % 3 == 1: y = (X[:, 2] <= 4).astype(int)
    else: y = (X[:, 0] > X[:, 1]).astype(int)
    table = pd.DataFrame(X, columns=['A', 'B', 'C']).assign(answer=y)
    return table, rule

seed = 7                        # ← change this number to get a new made-up table
table, rule = made_up_table(seed)
shown = table.astype({'answer': str}); shown.loc[len(table) - 1, 'answer'] = '?'
print('A made-up table. Can you guess the last answer?\n')
print(shown.to_string(index=False))

Got a guess? Run the next cell to see the hidden rule and the real answer.

In [ ]:
print(f'The hidden rule: {rule}')
print(f'The answer for the last row: {table["answer"].iloc[-1]}')

**Stop and notice:** you did not "train" anything. You **looked at the examples and spotted the pattern**. With a few made-up tables of practice, you probably got faster at it. Change `seed` above and try another one: you'll get better at the *skill*.

The foundation model did exactly this, **millions of times**, with much richer rules (curves, if-then rules, columns that matter only together, messy columns that don't matter at all). That is its experience.

## 5. How it predicts without training: your rows become the prompt

You know how to use ChatGPT with examples: *"Here are three customer emails and how we answered them. Now answer this fourth one."* ChatGPT does not retrain itself. It **reads the examples in the prompt** and continues the pattern. This is called **in-context learning** (the *ICL* in Tab**ICL**).

A tabular foundation model does the same thing with your table:

| | 🚀 Gradient boosting | 🧠 Tabular foundation model |
|---|---|---|
| `.fit(X, y)` | **studies** your rows and grows hundreds of trees | **just stores** your rows (they are the prompt) |
| `.predict(X_new)` | quick: walks down the trees | **reads** your rows + the new rows **in one pass** and answers |
| Dials to tune | learning rate, tree size, early stopping… | **none** |

And in code it looks exactly like every scikit-learn model you know:

```python
from tabicl import TabICLClassifier
model = TabICLClassifier()                 # no dials
model.fit(X_small, y_small)                # stores your bookings
model.predict_proba(X_new)[:, 1]           # reads them and answers
```

## 6. The race: Opening Day

Now the real test, on our hotel bookings. Same rules as tonight's honesty lesson:

- Both models get **exactly the same** bookings to learn from: **100**, then **300**, then **1,000**.
- Both are graded on **exactly the same** exam: 2,000 test bookings (2,000 instead of 6,000 to keep the race quick).
- Boosting gets its early stopping. TabICL gets **nothing**: no dials.

The first time, TabICL downloads its "brain" (the trained model file) from the internet: that takes a moment. On a T4 GPU the whole race takes about a minute; without a GPU it can take several minutes.

In [ ]:
exam_X, exam_y = X_te.iloc[:2000], y_te.iloc[:2000]          # the same exam for both models
race = []
try:
    from tabicl import TabICLClassifier
    for n in [100, 300, 1000]:
        X_small, y_small = first_bookings(n)                   # the same bookings for both models

        start = time.time()
        boost = new_boost().fit(X_small, y_small)
        boost_auc, boost_sec = roc_auc_score(exam_y, boost.predict_proba(exam_X)[:, 1]), time.time() - start

        start = time.time()
        tfm = TabICLClassifier().fit(X_small, y_small)        # no dials at all
        tfm_auc, tfm_sec = roc_auc_score(exam_y, tfm.predict_proba(exam_X)[:, 1]), time.time() - start

        race.append({'bookings': n, 'boosting AUC': boost_auc, 'TabICL AUC': tfm_auc, 'boosting seconds': boost_sec, 'TabICL seconds': tfm_sec})
        print(f'{n:>5,} bookings · boosting {boost_auc:.3f} · TabICL {tfm_auc:.3f}')
    race = pd.DataFrame(race).set_index('bookings')
    race['TabICL lead'] = race['TabICL AUC'] - race['boosting AUC']
    print('\n' + race.round(3).to_string())
except ImportError:
    print('tabicl is not installed yet. Run the very first code cell (pip install), then this cell again.')
except Exception as problem:
    print('TabICL could not run here:', problem)
    print('Most often this means the model file could not be downloaded, or the runtime ran out of memory. Try again on a T4 GPU runtime.')

The next cell draws the race on top of the Step 1 curve.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, [boost_curve[n] for n in sizes], 'o-', color='#5b6bd8', lw=2.5, ms=8, label='gradient boosting (starts from zero)')
if isinstance(race, pd.DataFrame) and len(race):
    ax.plot(race.index, race['TabICL AUC'], 's-', color='#1aa88f', lw=2.5, ms=9, label='TabICL (read millions of tables first)')
ax.axvspan(40, 600, color='#f0a030', alpha=0.12); ax.text(60, 0.97, 'Opening Day', color='#b06f00', va='top')
ax.set_xscale('log'); ax.set_xticks(sizes); ax.set_xticklabels([f'{n:,}' for n in sizes])
ax.set_ylim(0.6, 1.0); ax.set_xlabel('bookings the model learned from (log scale)'); ax.set_ylabel('test AUC')
ax.set_title('The fresh graduate vs. the experienced manager'); ax.legend(loc='lower right'); plt.show()

**How to read it:** the blue line is boosting from Step 1. The green squares are TabICL, which learned from the **same** small sets of bookings. (The blue line was graded on 6,000 test bookings, the green squares on 2,000 of them: close enough to compare by eye. The table above is the exact, same-exam comparison.)

**What to look for:**
- **Where is the gap biggest?** If the published benchmarks hold on our data, the experienced manager helps most when data is **scarce** (the Opening Day zone), and the gap **shrinks** as data grows. With tens of thousands of rows, boosting catches up: the fresh graduate has learned on the job.
- **Look at the seconds column.** Boosting trains and predicts in about a second on any laptop. TabICL is slower, especially without a GPU. Speed and cost are part of the decision.
- **Use the honesty rule from Session 9.** One small sample of bookings and one exam of 2,000 are partly luck. A lead of 0.005 either way is a tie. A lead of 0.03 is real.

### 🧠 Quick check

A teammate says: *"TabICL beat boosting with 300 bookings, so we should use it for everything from now on."* What would you ask?

<details><summary>Click for the answer</summary>

Three questions. **(1) How many rows will we really have?** With tens of thousands, boosting usually catches up. **(2) What does it cost to run?** It is slower and wants a GPU; boosting runs anywhere. **(3) Was the win real?** Check it with several samples or cross-validation, not one lucky exam. The right answer is not "always" or "never". It is **"try both, on the exam that looks like the job."**
</details>

## 7. So which one do I use?

| | 🚀 Gradient boosting | 🧠 Tabular foundation model |
|---|---|---|
| **Small data** (hundreds of rows) | struggles | ✅ **its sweet spot** |
| **Big data** (hundreds of thousands of rows and up) | ✅ fast and strong | possible with the newest models and a big GPU |
| **Tuning** | needs a few dials | ✅ none |
| **Speed and cost** | ✅ seconds, any laptop | slower, wants a GPU |
| **Scoring millions of rows every night** | ✅ cheap | can get expensive |
| **License** | ✅ free (scikit-learn, XGBoost, LightGBM) | TabICL: free and open. TabPFN's newest: **research only** unless you pay |
| **Explaining it** (Session 9) | ✅ SHAP works well | SHAP works too, but slower |

**The rule of thumb for 2026:** for a new problem, **try both**, judged on the exam that looks like the job. Small data, no time to tune → start with the foundation model. Big data, tight budget, millions of predictions a day → boosting is still the workhorse.

### 🧠 Quick check: five business cases

For each case, which would you try first? Think, then click.

1. A new boutique hotel, open for six weeks, **240 bookings**, wants a cancellation score by Friday.
2. A bank scores **5 million** card transactions **every night** for fraud, on a tight cloud budget.
3. A consulting firm has **180 clients** and wants to know which ones may not renew.
4. A startup wants to **sell** a product built on the model, and has no budget for licenses.
5. A hospital has **400 patient records** for a rare condition.

<details><summary>Click for the answers</summary>

1. **Foundation model first.** Tiny table, no time to tune: its sweet spot. Compare with boosting anyway.
2. **Boosting.** Huge volume, every night, tight budget: speed and cost win.
3. **Foundation model first.** 180 rows is exactly where boosting struggles.
4. **Boosting, or TabICL (open license).** Check every license before you sell anything built on a model. TabPFN's newest weights need a paid license for commercial use.
5. **Foundation model first**, judged very carefully: small data, high stakes. Use cross-validation, check the slices (Session 9), and keep a person in the loop.
</details>

## 8. Your turn

1. **Find the crossover.** Run the race at **2,000** and **5,000** bookings (change the list in the starter cell). Where does boosting catch up, if it does? What happens to the seconds?
2. **Is the win real?** Run the 300-booking race with three **different** sets of bookings (`sample=0, 1, 2`). Does the winner stay the same?
3. **The manager's question.** With only 300 bookings, what chance of canceling does each model give the guest from Session 9 (booking 438: booked 312 days ahead, no special requests, online travel agency; the guest **canceled**)? With 24,000 bookings our tuned model said **87%**.

In [ ]:
# Q1 · find the crossover
more_sizes = [2000, 5000]          # ← your turn
try:
    for n in more_sizes:
        X_small, y_small = first_bookings(n)
        start = time.time(); b = new_boost().fit(X_small, y_small); b_auc, b_sec = roc_auc_score(exam_y, b.predict_proba(exam_X)[:, 1]), time.time() - start
        start = time.time(); t = TabICLClassifier().fit(X_small, y_small); t_auc, t_sec = roc_auc_score(exam_y, t.predict_proba(exam_X)[:, 1]), time.time() - start
        print(f'{n:>5,} bookings · boosting {b_auc:.3f} ({b_sec:.0f}s) · TabICL {t_auc:.3f} ({t_sec:.0f}s)')
except Exception as problem:
    print('Run Step 6 first (it needs TabICL):', problem)

In [ ]:
# Q2 · is the win real? Three different sets of 300 bookings
try:
    for sample in [0, 1, 2]:       # ← your turn: try other numbers too
        X_small, y_small = first_bookings(300, sample)
        b_auc = roc_auc_score(exam_y, new_boost().fit(X_small, y_small).predict_proba(exam_X)[:, 1])
        t_auc = roc_auc_score(exam_y, TabICLClassifier().fit(X_small, y_small).predict_proba(exam_X)[:, 1])
        print(f'sample {sample}: boosting {b_auc:.3f} · TabICL {t_auc:.3f} · lead {t_auc - b_auc:+.3f}')
except Exception as problem:
    print('Run Step 6 first (it needs TabICL):', problem)

In [ ]:
# Q3 · the manager's booking (test booking number 438, the one from Session 9's receipt)
guest = X_te.iloc[[438]]
print(f'What really happened: {"CANCELED" if y_te.iloc[438] == 1 else "showed up"}\n')
try:
    X_small, y_small = first_bookings(300)
    print(f'boosting, 300 bookings: {new_boost().fit(X_small, y_small).predict_proba(guest)[0, 1]:.0%} chance of canceling')
    print(f'TabICL,   300 bookings: {TabICLClassifier().fit(X_small, y_small).predict_proba(guest)[0, 1]:.0%} chance of canceling')
except Exception as problem:
    print('Run Step 6 first (it needs TabICL):', problem)

Your answers (double-click to edit):

- **Q1:** *boosting catches up at about \_\_\_ bookings. TabICL took \_\_\_ seconds there, boosting \_\_\_.*
- **Q2:** *the winner stayed the same / changed. The lead moved between \_\_\_ and \_\_\_, so I call it real / a tie.*
- **Q3:** *with 300 bookings, boosting said \_\_\_% and TabICL said \_\_\_%. The guest canceled, so \_\_\_ was closer.*

## What we learned tonight

- **Every model so far starts from zero**, so it needs a lot of data. On our hotel bookings, boosting scores about **0.80 with 200 bookings** and needs **thousands** to reach 0.93.
- **Small data is the normal case in business**: new stores, new products, B2B clients, rare events, expensive labels.
- A **tabular foundation model** (TabICL, TabPFN) has **read millions of made-up tables** first. It did not learn hotel facts: **it learned how to learn from a table**. It is the experienced manager.
- It predicts by **in-context learning**: `.fit()` stores your rows, `.predict()` reads them like a prompt and answers in one pass. **No dials.**
- **Which one?** Try both, on the exam that looks like the job. Foundation models shine on **small** tables with no time to tune. Boosting is still the workhorse for **big, fast, cheap**. And **check the license** before you build a business on a model.

### Tonight's words, and their real names

| What we said | The real name |
|---|---|
| the model that already read millions of tables | **tabular foundation model** (TFM) |
| its made-up textbooks | **synthetic pretraining data** (the "prior") |
| your rows become the prompt | **in-context learning** (ICL) |
| a new hotel with little data | the **cold-start** or **small-data** problem |
| how the score grows with more data | a **learning curve** |